# Visualisation du prompt envoyé au LLM (1ère requête Gold test)

Affiche à quoi ressemble le **prompt complet** (tous les messages) envoyé au LLM pour la **première requête** du gold test, pour chaque stratégie few-shot : **Fewshot** (type+section), **Fewshot_Kate** (1E+1C sémantique), **Fewshot_Cosine** (top-k cosine).

## 1. Chemins vers les JSONL few-shot

In [2]:
import json
from pathlib import Path

cwd = Path(".").resolve()
if (cwd.parent.parent / "Gold_test.json").exists():
    NLI4CT_ROOT = cwd.parent.parent
elif (cwd.parent / "Gold_test.json").exists():
    NLI4CT_ROOT = cwd.parent
else:
    NLI4CT_ROOT = cwd

RESULTS = NLI4CT_ROOT / "results"
FILES = {
    "Fewshot (type+section)": RESULTS / "Fewshot" / "Gold_test_fewshot_prompt1.jsonl",
    "Fewshot_Kate (1E+1C)": RESULTS / "Fewshot_Kate" / "Gold_test_fewshot_kate_prompt1.jsonl",
    "Fewshot_Cosine": RESULTS / "Fewshot_Cosine" / "Gold_test_fewshot_cosine_prompt1.jsonl",
}
MAX_CHARS = 600  # troncature pour l'affichage (0 = tout afficher)

for name, p in FILES.items():
    status = "OK" if p.exists() else "MANQUANT"
    print(f"{name}: {status} -> {p}")

Fewshot (type+section): OK -> /Users/lubin/Documents/NLI_Finetuning/NLI4CT/results/Fewshot/Gold_test_fewshot_prompt1.jsonl
Fewshot_Kate (1E+1C): OK -> /Users/lubin/Documents/NLI_Finetuning/NLI4CT/results/Fewshot_Kate/Gold_test_fewshot_kate_prompt1.jsonl
Fewshot_Cosine: OK -> /Users/lubin/Documents/NLI_Finetuning/NLI4CT/results/Fewshot_Cosine/Gold_test_fewshot_cosine_prompt1.jsonl


## 2. Fonction d'affichage d'une entrée

In [3]:
def truncate(s, max_chars=600):
    if max_chars and len(s) > max_chars:
        return s[:max_chars] + "..."
    return s


def show_first_request(jsonl_path, strategy_name, max_chars=600):
    if not jsonl_path.exists():
        print(f"[Fichier absent: {jsonl_path}]")
        return
    with open(jsonl_path, "r", encoding="utf-8") as f:
        first = json.loads(f.readline())
    messages = first["messages"]
    print("=" * 70)
    print(f"  {strategy_name}")
    print("=" * 70)
    print(f"Nombre de messages: {len(messages)}\n")
    for i, m in enumerate(messages):
        role = m["role"].upper()
        content = truncate(m["content"], max_chars)
        print(f"--- [{role}] ---")
        print(content)
        print()
    print()

## 3. Affichage pour chaque stratégie (1ère requête)

In [4]:
for name, path in FILES.items():
    show_first_request(path, name, max_chars=MAX_CHARS)

  Fewshot (type+section)
Nombre de messages: 11

--- [SYSTEM] ---
You will see several examples of premise-hypothesis pairs with their classification (Entailment or Contradiction). Use these examples to understand the task. Then classify the relationship between the last premise and hypothesis. Respond with only one word: 'Entailment' or 'Contradiction'.

--- [USER] ---
PREMISE: Inclusion Criteria:
  Body mass index (BMI) of 19 to 40 kg/m^2 , inclusive

Inclusion Criteria:
  Patients must have histologically-confirmed adenocarcinoma of the breast with operable or inoperable stage 1c (primary tumor > 1.0 cm), II or III disease.
  Measurable disease by physical examinations or diagnostic breast imaging (mammography, ultrasonography or MR).
  Pre-treatment core or incisional biopsy. Patients may not have had definitive primary surgery.
  Male or female, 18 years of age or older.
  ECOG performance status 0 or 1.
  Adequate organ function as defined in the protoc...

--- [ASSISTANT] ---
Co

## 4. Détail : dernière paire user/assistant (requête à évaluer + gold)

Pour une stratégie donnée, on affiche en entier le **dernier** message user (la requête à classifier) et le **dernier** assistant (réponse gold).

In [5]:
strategy = "Fewshot_Cosine"  # modifier si besoin
path = FILES.get(strategy) or FILES["Fewshot_Cosine"]
if path.exists():
    with open(path, "r", encoding="utf-8") as f:
        first = json.loads(f.readline())
    msgs = first["messages"]
    print("Dernier USER (requête à classifier) — affiché en entier:")
    print("-" * 50)
    print(msgs[-2]["content"])
    print("-" * 50)
    print("Dernier ASSISTANT (réponse gold):", msgs[-1]["content"])

Dernier USER (requête à classifier) — affiché en entier:
--------------------------------------------------
PREMISE: Exclusion Criteria:
  History of bilateral mastectomy, osteoporosis or renal impairment.

Exclusion Criteria:
  Severe claustrophobia

HYPOTHESIS: Women suffering from both claustrophobia and IBS or not eligible for either the primary trial or the secondary trial.
--------------------------------------------------
Dernier ASSISTANT (réponse gold): Contradiction
